# 10 · BERT — COVID tweet classifier

Fine-tune `bert-base-cased` on the course COVID Twitter CSV.

Source: `Transformers_Exercise`

**Colab GPU strongly recommended.** 3 epochs on the full 5.2k rows is a few minutes on T4. Set `MAX_SAMPLES` if you want a smoke test.

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

try:
    from transformers import BertForSequenceClassification, AutoTokenizer
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])
    from transformers import BertForSequenceClassification, AutoTokenizer

In [ ]:
CSV_NAME = 'covid_twitter_dataset_codebasics_DL.csv'
RAW = 'https://raw.githubusercontent.com/yash9614/lear_deepintodl/main/Transformers_Exercise/covid_twitter_dataset_codebasics_DL.csv'
LOCALS = [CSV_NAME, f'../Transformers_Exercise/{CSV_NAME}', f'/content/{CSV_NAME}']

def load_csv(raw, locals_, name):
    try:
        return pd.read_csv(raw)
    except Exception:
        pass
    for p in locals_:
        if os.path.exists(p):
            return pd.read_csv(p)
    from google.colab import files
    print('Upload', name)
    up = files.upload()
    return pd.read_csv(next(iter(up)))

data = load_csv(RAW, LOCALS, CSV_NAME)
print(data.head())
print(data.info())
print(data['target'].value_counts())

MAX_SAMPLES = None  # e.g. 1500 for a quicker dry run
if MAX_SAMPLES:
    data = data.sample(n=min(MAX_SAMPLES, len(data)), random_state=42)

In [ ]:
train_X, test_X, train_Y, test_Y = train_test_split(
    data['text'].astype(str), data['target'].astype(int),
    test_size=0.30, random_state=42, stratify=data['target'],
)

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
train_tokens = tokenizer(list(train_X), padding=True, truncation=True, max_length=128)
test_tokens = tokenizer(list(test_X), padding=True, truncation=True, max_length=128)
print(train_tokens.keys())
print(train_tokens['input_ids'][0][:20])
print(tokenizer.decode(train_tokens['input_ids'][0][:30]))

In [ ]:
class TokenData(Dataset):
    def __init__(self, train=False):
        if train:
            self.text_data = train_X
            self.tokens = train_tokens
            self.labels = list(train_Y)
        else:
            self.text_data = test_X
            self.tokens = test_tokens
            self.labels = list(test_Y)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sample = {k: torch.tensor(v[idx]) for k, v in self.tokens.items()}
        sample['labels'] = torch.tensor(self.labels[idx])
        return sample

batch_size = 30
train_dataset = TokenData(train=True)
test_dataset = TokenData(train=False)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print(len(train_dataset), len(test_dataset), next(iter(train_loader)).keys())

In [ ]:
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-cased', num_labels=2
)
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=1e-5)
loss_fn = nn.CrossEntropyLoss()
num_epochs = 3
bert_model.to(device)

In [ ]:
for epoch in range(num_epochs):
    bert_model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = bert_model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        pred = outputs.logits
        loss = loss_fn(pred, batch['labels'])
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{num_epochs} - Training Loss: {avg_train_loss:.4f}')

In [ ]:
bert_model.eval()
correct = total = 0
total_test_loss = 0.0
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = bert_model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        logits = outputs.logits
        loss = loss_fn(logits, batch['labels'])
        total_test_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == batch['labels']).sum().item()
        total += batch['labels'].size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

avg_test_loss = total_test_loss / len(test_loader)
accuracy = 100 * correct / total
print(f'Test Set - Loss: {avg_test_loss:.4f}, Accuracy: {accuracy:.2f}%')

### Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['not covid', 'covid'], yticklabels=['not covid', 'covid'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('BERT COVID tweets')
plt.show()

### Inference

In [ ]:
test_tweets = [
    'The number of COVID-19 cases has increased in the past week.',
    'Just watched an amazing sunset at the beach!',
    'New vaccination centers are opening up to combat the spread of coronavirus.',
    'A virus has impacted the global ecosystem a lot',
    'India lost the match againt Australia at Adelaide',
]

def predict_tweet(tweet_text):
    tokens = tokenizer(tweet_text, padding=True, truncation=True, return_tensors='pt')
    tokens = {k: v.to(device) for k, v in tokens.items()}
    bert_model.eval()
    with torch.no_grad():
        outputs = bert_model(**tokens)
        probs = torch.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probs, dim=1)
    return {
        'text': tweet_text,
        'is_covid': bool(prediction.item()),
        'confidence': probs[0][prediction[0]].item() * 100,
    }

print('Sample Tweet Predictions:\n')
for tweet in test_tweets:
    result = predict_tweet(tweet)
    print(f"Tweet: {result['text']}")
    print(f"Prediction: {'COVID-related' if result['is_covid'] else 'Not COVID-related'}")
    print(f"Confidence: {result['confidence']:.2f}%")
    print('-' * 50 + '\n')